# Static UMAP Plotting with Raw Merger Time-Since Flags

This notebook is a cleaned copy of `static_umap_plotting_w_xmatched_samples_single_flag.ipynb`.
The overlay selections use the raw merger-catalog columns that TNG Tools already appends to `catalog.fits`, rather than the custom renamed boolean labels.



## Workflow

1. Load UMAP coordinates and the object IDs stored in the Hyrax inference metadata.
2. Load the run/sample `catalog.fits` file.
3. Validate that the raw merger time-since column is present in that catalog.
4. Plot the full UMAP in gray and overlay the objects whose last merger falls inside each time window.

The default setup uses `Major_TimeSinceMerger` and makes three columns per experiment: major mergers within the last 0.75, 1.0, and 1.25 Gyr. Missing/no-merger values are usually stored as `-1`, so each default selection requires `0 <= time_since_merger <= window`.



## Imports

Keep the imports in one place so the rest of the notebook only contains data-loading and plotting logic.


In [2]:
from __future__ import annotations

import logging
import re
from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import hyrax
from IPython.display import display


## Paths, Catalog Defaults, and Time-Window Configuration

Adjust these paths if your run outputs live somewhere else. The catalog paths should point to the TNG Tools `catalog.fits` outputs that already include the raw merger-catalog columns.

By default, every run uses the regular full catalog (`catalog_key='all'`). To use a cutout-size split for a one-off plot, pass `catalog_key='le_120x120'` or `catalog_key='gt_120x120'` to `plot_run_raw_merger_flags(...)`. If a future run should always use a split catalog, add only that run to `RUN_CATALOG_OVERRIDES`.

Use `build_time_since_merger_flags(...)` to switch merger type or time windows. For example, `build_time_since_merger_flags('Minor', (0.5, 1.0, 1.5))` will create three minor-merger panels per experiment.



In [3]:
def first_existing_path(candidates: Iterable[str | Path]) -> Path:
    """Return the first existing path, or the first candidate for readable errors."""
    paths = [Path(candidate).expanduser() for candidate in candidates]
    for path in paths:
        if path.exists():
            return path
    return paths[0]


PROJECT_ROOT = Path.cwd()

HYRAX_RUN_BASE = Path('/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs')

CATALOG_PATHS = {
    'all': first_existing_path([
        '/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/split_images/catalog.fits',
        PROJECT_ROOT / 'data' / 'catalog.fits',
    ]),
    'le_120x120': first_existing_path([
        '/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/split_images/catalog_le_120x120.fits',
        PROJECT_ROOT / 'data' / 'catalog_le_120x120.fits',
    ]),
    'gt_120x120': first_existing_path([
        '/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/split_images/catalog_gt_120x120.fits',
        PROJECT_ROOT / 'data' / 'catalog_gt_120x120.fits',
    ]),
}

DEFAULT_CATALOG_KEY = 'all'

# Optional: add only exceptional run-specific defaults here.
# Example: RUN_CATALOG_OVERRIDES = {12: 'le_120x120', 13: 'gt_120x120'}
RUN_CATALOG_OVERRIDES = {}


def resolve_catalog_key(run: int, catalog_key: str | None = None) -> str:
    """Resolve the catalog choice for a run, defaulting all runs to the full catalog."""
    resolved = catalog_key or RUN_CATALOG_OVERRIDES.get(run, DEFAULT_CATALOG_KEY)
    if resolved not in CATALOG_PATHS:
        valid = ', '.join(sorted(CATALOG_PATHS))
        raise KeyError(f"Unknown catalog_key '{resolved}'. Choose one of: {valid}")
    return resolved


MERGER_TYPE_STYLES = {
    'Major': {'color': 'tab:red', 'marker': 'x'},
    'Minor': {'color': 'tab:green', 'marker': 's'},
    'Mini': {'color': 'tab:blue', 'marker': '.'},
}

DEFAULT_TIME_WINDOWS_GYR = (0.75, 1.0, 1.25)


def build_time_since_merger_flags(
    merger_type: str = 'Major',
    windows_gyr: Iterable[float] = DEFAULT_TIME_WINDOWS_GYR,
    min_time_gyr: float = 0.0,
) -> list[dict]:
    """Build one overlay flag per time-since-merger window.

    The raw catalog columns are named like `Major_TimeSinceMerger`.
    `min_time_gyr=0` excludes the catalog's usual no-merger sentinel value of -1.
    """
    merger_type = merger_type.strip().capitalize()
    style = MERGER_TYPE_STYLES.get(merger_type, {'color': 'tab:red', 'marker': 'x'})
    key = f'{merger_type}_TimeSinceMerger'

    flags = []
    for window in windows_gyr:
        window = float(window)
        flags.append({
            'key': key,
            'min_value': float(min_time_gyr),
            'max_value': window,
            'include_min': True,
            'include_max': True,
            'color': style['color'],
            'marker': style['marker'],
            'label': f'{merger_type} merger <= {window:g} Gyr',
            'summary_label': f'{merger_type}_TimeSinceMerger <= {window:g} Gyr',
        })
    return flags


RAW_MERGER_FLAGS = build_time_since_merger_flags(
    merger_type='Major',
    windows_gyr=DEFAULT_TIME_WINDOWS_GYR,
)



## Catalog Loading and ID Normalization

The UMAP metadata and FITS catalogs can store IDs as strings, integers, or byte strings. These helpers normalize them before matching catalog rows back to UMAP points.


In [4]:
def load_external_catalog(catalog_path: str | Path) -> pd.DataFrame:
    """Load an image/sample catalog from FITS, parquet, or CSV."""
    catalog_path = Path(catalog_path)
    suffix = catalog_path.suffix.lower()

    if suffix in {'.parquet', '.pq'}:
        return pd.read_parquet(catalog_path)

    if suffix in {'.csv', '.txt'}:
        return pd.read_csv(catalog_path)

    if suffix in {'.fits', '.fit', '.fts'}:
        from astropy.table import Table

        return Table.read(catalog_path).to_pandas()

    raise ValueError(f"Unsupported catalog format '{suffix}'. Use FITS, parquet, or CSV.")


def _decode_scalar(value):
    """Decode byte strings from FITS tables while leaving other values unchanged."""
    if isinstance(value, (bytes, bytearray)):
        return value.decode('utf-8').strip()
    return value


def normalize_object_ids(values) -> pd.Series:
    """Normalize object IDs to comparable nullable strings."""
    ids = pd.Series(values, copy=False).map(_decode_scalar)
    missing = ids.isna()
    numeric = pd.to_numeric(ids, errors='coerce')

    if (~missing).any() and numeric.loc[~missing].notna().all():
        normalized = numeric.astype('Int64').astype('string')
    else:
        normalized = ids.astype('string').str.strip()

    return normalized.mask(missing)


def resolve_catalog_id_column(catalog: pd.DataFrame, catalog_id_column: str | None = None) -> str:
    """Find the catalog object-ID column used to match rows back to UMAP metadata."""
    candidates = [
        catalog_id_column,
        'object_id',
        'rubin_object_id',
        'objectId',
        'objectId_data',
        'id',
    ]

    for candidate in candidates:
        if candidate is not None and candidate in catalog.columns:
            return candidate

    raise KeyError(
        'Could not find an object ID column in the catalog. '
        'Pass catalog_id_column explicitly.'
    )


## Raw Merger Columns in `catalog.fits`

TNG Tools appends the raw merger-catalog fields directly to the FITS catalog. The notebook only needs to confirm that the requested time-since-merger column is present and add a normalized `_match_id` for UMAP matching.



In [5]:
def raw_columns_from_flags(flags: list[dict]) -> list[str]:
    """Return unique raw catalog columns requested by the flag configuration."""
    return list(dict.fromkeys(flag['key'] for flag in flags))


def prepare_catalog_with_raw_merger_flags(
    catalog: pd.DataFrame,
    flags: list[dict],
    catalog_id_column: str | None = None,
) -> pd.DataFrame:
    """Validate raw merger columns in `catalog.fits` and add a UMAP match key."""
    catalog = catalog.copy()
    raw_columns = raw_columns_from_flags(flags)
    missing = [column for column in raw_columns if column not in catalog.columns]

    if missing:
        raise KeyError(
            'The loaded catalog is missing these raw merger columns: '
            f'{missing}. Check that CATALOG_PATHS points to the TNG Tools '
            'catalog with appended merger-catalog fields.'
        )

    catalog_id_column = resolve_catalog_id_column(catalog, catalog_id_column)
    catalog['_match_id'] = normalize_object_ids(catalog[catalog_id_column])
    return catalog


def describe_flag_selection(flag: dict) -> str:
    """Build a compact label for the numeric selection represented by one flag."""
    if 'summary_label' in flag:
        return flag['summary_label']

    key = flag['key']
    if 'min_value' in flag or 'max_value' in flag:
        min_value = flag.get('min_value')
        max_value = flag.get('max_value')
        include_min = flag.get('include_min', True)
        include_max = flag.get('include_max', True)
        lower = '<=' if include_min else '<'
        upper = '<=' if include_max else '<'

        if min_value is not None and max_value is not None:
            return f'{min_value:g} {lower} {key} {upper} {max_value:g}'
        if max_value is not None:
            return f'{key} {upper} {max_value:g}'
        if min_value is not None:
            return f'{min_value:g} {lower} {key}'

    threshold = flag.get('threshold', 1)
    comparator = flag.get('comparator', '>=')
    return f'{key} {comparator} {threshold}'


def summarize_catalog_flags(catalog: pd.DataFrame, flags: list[dict]) -> pd.DataFrame:
    """Return a compact count table for the raw merger selections used here."""
    summary = {
        'catalog_rows': len(catalog),
        'catalog_rows_with_match_id': int(catalog['_match_id'].notna().sum()),
    }

    for flag in flags:
        values = pd.to_numeric(catalog[flag['key']], errors='coerce')
        selected = _flag_mask(values, flag)
        summary[describe_flag_selection(flag)] = int(selected.fillna(False).sum())

    return pd.DataFrame([summary])



## Loading UMAP Results

The plotting functions need both UMAP coordinates and the object IDs that Hyrax stored alongside each point. The metadata reader tries several common object-ID field names and reports what it used.


In [6]:
def resolve_umap_paths(
    run: int,
    expt: int,
    run_base: str | Path = HYRAX_RUN_BASE,
) -> tuple[Path, Path]:
    """Read the run/expt log and return the UMAP results directory plus config file."""
    run_dir = Path(run_base) / f'run{run}'
    run_name = f'udb{run}_{expt}'
    log_path = run_dir / f'{run_name}.txt'
    config_path = run_dir / f'{run_name}.toml'

    content = log_path.read_text()
    match = re.search(r'Saving UMAP results to (.+)', content)
    if match is None:
        raise ValueError(f'Could not find UMAP results directory in {log_path}')

    return Path(match.group(1).strip()), config_path


def get_umap_with_ids(
    config=None,
    input_dir: str | Path | None = None,
    suppress_logs: bool = True,
    id_field: str = 'objectId_data',
) -> dict:
    """Load UMAP coordinates and the object IDs used for catalog matching."""
    from hyrax.data_sets.inference_dataset import InferenceDataSet

    def extract_metadata_column(metadata_obj, field_name: str):
        """Extract one metadata field from dict, DataFrame, structured array, or array payloads."""
        if isinstance(metadata_obj, dict):
            if field_name in metadata_obj:
                return np.asarray(metadata_obj[field_name])
            if len(metadata_obj) == 1:
                return np.asarray(next(iter(metadata_obj.values())))
            return None

        if hasattr(metadata_obj, 'columns'):
            columns = list(metadata_obj.columns)
            if field_name in columns:
                return metadata_obj[field_name].to_numpy()
            if len(columns) == 1:
                return metadata_obj[columns[0]].to_numpy()
            return None

        dtype = getattr(metadata_obj, 'dtype', None)
        names = getattr(dtype, 'names', None)
        if names:
            if field_name in names:
                return np.asarray(metadata_obj[field_name])
            if len(names) == 1:
                return np.asarray(metadata_obj[names[0]])
            return None

        array = np.asarray(metadata_obj)
        if array.ndim == 1:
            return array
        if array.ndim == 2 and array.shape[1] == 1:
            return array[:, 0]
        return None

    if suppress_logs:
        logging.disable(logging.CRITICAL)

    umap_results = InferenceDataSet(config, results_dir=input_dir, verb='umap')

    logging.disable(logging.NOTSET)

    points = np.array([point.numpy() for point in umap_results])
    x, y = points[:, 0], points[:, 1]

    all_indices = list(range(len(umap_results)))
    available_fields = list(umap_results.metadata_fields())
    preferred_fields = [
        id_field,
        'objectId_data',
        'object_id_data',
        'objectId',
        'object_id',
        'rubin_object_id',
        'id',
    ]

    candidate_fields = []
    for field in preferred_fields:
        if field is not None and field not in candidate_fields:
            candidate_fields.append(field)

    candidate_fields = [field for field in candidate_fields if field in available_fields] + [
        field for field in candidate_fields if field not in available_fields
    ]

    attempts = []
    rubin_ids = None
    resolved_field = None

    for candidate in candidate_fields:
        try:
            metadata = umap_results.metadata(all_indices, [candidate])
            extracted = extract_metadata_column(metadata, candidate)
            if extracted is None:
                attempts.append(f'{candidate}: not found in metadata payload')
                continue
            if len(extracted) != len(umap_results):
                attempts.append(f'{candidate}: length mismatch')
                continue

            rubin_ids = np.asarray(extracted)
            resolved_field = candidate
            break
        except Exception as exc:
            attempts.append(f'{candidate}: {exc}')

    if rubin_ids is None:
        raise KeyError(
            'Could not extract object IDs from UMAP metadata. Attempts: '
            + '; '.join(attempts)
        )

    return {
        'x': x,
        'y': y,
        'rubin_ids': rubin_ids,
        'id_field': resolved_field,
        'umap_results': umap_results,
    }


def load_umap_for_run_expt(
    run: int,
    expt: int,
    run_base: str | Path = HYRAX_RUN_BASE,
    suppress_logs: bool = True,
) -> dict:
    """Load one run/expt pair using the Hyrax config referenced by the run log."""
    umap_dir, config_file = resolve_umap_paths(run, expt, run_base=run_base)

    if suppress_logs:
        logging.disable(logging.CRITICAL)
    h = hyrax.Hyrax(config_file=config_file)
    logging.disable(logging.NOTSET)

    return get_umap_with_ids(
        config=h.config,
        input_dir=umap_dir,
        suppress_logs=suppress_logs,
    )


## Matching Time-Since-Merger Selections to UMAP Points

Each time-since window is filtered directly from `catalog.fits`, then the selected object IDs are merged onto the UMAP coordinates. The selected IDs are de-duplicated before the merge so repeated filters for the same object do not create a Cartesian product.



In [7]:
def _comparison_mask(values: pd.Series, threshold: float, comparator: str) -> pd.Series:
    """Apply a simple numeric threshold to a raw merger column."""
    if comparator == '>=':
        return values >= threshold
    if comparator == '>':
        return values > threshold
    if comparator == '<=':
        return values <= threshold
    if comparator == '<':
        return values < threshold
    if comparator == '==':
        return values == threshold
    raise ValueError(f"Unsupported comparator '{comparator}'")


def _range_mask(values: pd.Series, flag: dict) -> pd.Series:
    """Apply a min/max range cut such as 0 <= time_since_merger <= 1.0."""
    mask = values.notna()

    min_value = flag.get('min_value')
    if min_value is not None:
        if flag.get('include_min', True):
            mask &= values >= min_value
        else:
            mask &= values > min_value

    max_value = flag.get('max_value')
    if max_value is not None:
        if flag.get('include_max', True):
            mask &= values <= max_value
        else:
            mask &= values < max_value

    return mask


def _flag_mask(values: pd.Series, flag: dict) -> pd.Series:
    """Apply either a range-style flag or a comparator-style flag."""
    if 'min_value' in flag or 'max_value' in flag:
        return _range_mask(values, flag)

    threshold = flag.get('threshold', 1)
    comparator = flag.get('comparator', '>=')
    return _comparison_mask(values, threshold, comparator)


def match_catalog_to_umap(
    umap_data: dict,
    catalog_with_flags: pd.DataFrame,
    flag: dict,
) -> pd.DataFrame:
    """Return UMAP coordinates whose catalog rows pass one raw merger cut."""
    key = flag['key']
    if key not in catalog_with_flags.columns:
        raise KeyError(
            f"Raw merger column '{key}' not found. Available columns include: "
            f"{list(catalog_with_flags.columns[:20])}"
        )

    values = pd.to_numeric(catalog_with_flags[key], errors='coerce')
    selected = catalog_with_flags.loc[
        _flag_mask(values, flag).fillna(False),
        ['_match_id', key],
    ].copy()

    if selected.empty:
        return pd.DataFrame(columns=['x', 'y', key])

    selected = selected.dropna(subset=['_match_id']).drop_duplicates('_match_id')
    umap_lookup = pd.DataFrame({
        'x': umap_data['x'],
        'y': umap_data['y'],
        '_match_id': normalize_object_ids(umap_data['rubin_ids']),
    }).dropna(subset=['_match_id'])

    return selected.merge(umap_lookup, on='_match_id', how='inner')


def plot_umap_flag_overlay(
    ax,
    umap_data: dict,
    catalog_with_flags: pd.DataFrame,
    flag: dict,
    alpha_background: float = 0.5,
    s_background: float = 1,
    title: str | None = None,
    show_legend: bool = True,
) -> pd.DataFrame:
    """Plot one UMAP panel with one raw merger-column overlay."""
    key = flag['key']
    color = flag.get('color', 'tab:red')
    marker = flag.get('marker', 'x')
    label = flag.get('label', describe_flag_selection(flag))
    alpha = flag.get('alpha', 0.6)
    size = flag.get('s', 5)

    ax.scatter(
        umap_data['x'],
        umap_data['y'],
        alpha=alpha_background,
        s=s_background,
        c='gray',
        label='All UMAP points',
    )

    matched = match_catalog_to_umap(
        umap_data,
        catalog_with_flags,
        flag,
    )

    if not matched.empty:
        ax.scatter(
            matched['x'].to_numpy(),
            matched['y'].to_numpy(),
            alpha=alpha,
            s=size,
            c=color,
            marker=marker,
            label=f'{label} (n={len(matched)})',
        )

    if title:
        ax.set_title(title)
    if show_legend:
        ax.legend(loc='best', fontsize='small')

    return matched



## Plotting Grids

These wrappers mirror the old notebook: one row per experiment and one column per raw merger flag.


In [8]:
def plot_umap_merger_flag_grid(
    run: int,
    expts,
    catalog_with_flags: pd.DataFrame,
    flags: list[dict],
    run_base: str | Path = HYRAX_RUN_BASE,
    figsize: tuple | None = None,
    dpi: int = 150,
    save_path: str | Path | None = None,
    suptitle: str | None = None,
    suppress_logs: bool = True,
    alpha_background: float = 0.5,
    s_background: float = 1,
    show_legend: bool = True,
):
    """Plot one row per experiment and one column per raw merger flag."""
    try:
        from tqdm.notebook import tqdm
    except Exception:
        tqdm = lambda iterable, total=None: iterable

    expts = list(expts)
    nrows = len(expts)
    ncols = len(flags)

    if nrows == 0:
        raise ValueError('At least one experiment is required.')
    if ncols == 0:
        raise ValueError('At least one flag configuration is required.')
    if figsize is None:
        figsize = (ncols * 4, nrows * 3)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=figsize,
        dpi=dpi,
        squeeze=False,
    )

    for row, expt in enumerate(tqdm(expts, total=nrows)):
        try:
            umap_data = load_umap_for_run_expt(
                run,
                expt,
                run_base=run_base,
                suppress_logs=suppress_logs,
            )
        except Exception as exc:
            for col in range(ncols):
                ax = axes[row, col]
                ax.text(
                    0.5,
                    0.5,
                    f'Error loading\nRun {run}, Expt {expt}\n{exc}',
                    ha='center',
                    va='center',
                    transform=ax.transAxes,
                )
                ax.set_title(f'Run {run}, Expt {expt}')
            continue

        for col, flag in enumerate(flags):
            ax = axes[row, col]
            label = flag.get('label', describe_flag_selection(flag))
            title = f'Run {run}, Expt {expt}: {label}'

            try:
                plot_umap_flag_overlay(
                    ax,
                    umap_data,
                    catalog_with_flags,
                    flag,
                    alpha_background=alpha_background,
                    s_background=s_background,
                    title=title,
                    show_legend=show_legend,
                )
            except Exception as exc:
                ax.text(
                    0.5,
                    0.5,
                    str(exc),
                    ha='center',
                    va='center',
                    transform=ax.transAxes,
                )
                ax.set_title(title)

    if suptitle:
        fig.suptitle(suptitle, fontsize=16)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=dpi, bbox_inches='tight')
    else:
        plt.show()

    return fig, axes


def plot_run_raw_merger_flags(
    run: int,
    expts=range(1, 9),
    flags=RAW_MERGER_FLAGS,
    catalog_key: str | None = None,
    catalog_id_column: str | None = None,
    run_base: str | Path = HYRAX_RUN_BASE,
    **kwargs,
):
    """Load the selected catalog with appended raw merger columns and plot the run grid."""
    resolved_catalog_key = resolve_catalog_key(run, catalog_key=catalog_key)
    catalog = load_external_catalog(CATALOG_PATHS[resolved_catalog_key])
    catalog_with_flags = prepare_catalog_with_raw_merger_flags(
        catalog=catalog,
        flags=flags,
        catalog_id_column=catalog_id_column,
    )

    display(summarize_catalog_flags(catalog_with_flags, flags))

    return plot_umap_merger_flag_grid(
        run=run,
        expts=expts,
        catalog_with_flags=catalog_with_flags,
        flags=flags,
        run_base=run_base,
        suptitle=f'Run {run} ({resolved_catalog_key}, raw time-since-merger columns from catalog.fits)',
        **kwargs,
    )



## Quick Catalog Sanity Check

Run this before plotting if you want to verify that the resolved `catalog.fits` contains the raw time-since-merger column used by this notebook.



In [9]:
catalog_key = 'all'
catalog = load_external_catalog(CATALOG_PATHS[catalog_key])
raw_columns = raw_columns_from_flags(RAW_MERGER_FLAGS)
available_raw_columns = [column for column in raw_columns if column in catalog.columns]
missing_raw_columns = [column for column in raw_columns if column not in catalog.columns]

print(CATALOG_PATHS[catalog_key])
print(f'Available raw merger columns: {available_raw_columns}')
if missing_raw_columns:
    print(f'Missing raw merger columns: {missing_raw_columns}')

preview_columns = [resolve_catalog_id_column(catalog), *available_raw_columns]
catalog[preview_columns].head()



/Users/diegomiura/research/Hyrax-Research/data/catalog.fits
Available raw merger columns: []
Missing raw merger columns: ['Major_TimeSinceMerger']


,object_id
0,72000000
1,72000000
2,72000000
3,72000000
4,72000000


## Example: Run 2 Major Merger Time Windows

This makes three columns per experiment: major mergers within the last 0.75, 1.0, and 1.25 Gyr.



In [10]:
fig, axes = plot_run_raw_merger_flags(
    run=2,
    expts=range(1, 9),
    flags=RAW_MERGER_FLAGS,
    alpha_background=0.5,
    show_legend=True,
)



KeyError: "The loaded catalog is missing these raw merger columns: ['Major_TimeSinceMerger']. Check that CATALOG_PATHS points to the TNG Tools catalog with appended merger-catalog fields."

## Optional: Change Merger Type, Time Windows, or Catalog Split

Use the same plotting code for minor or mini mergers, for a different set of lookback-time windows, or for the `le_120x120` / `gt_120x120` catalog splits.



In [ ]:
minor_time_windows = build_time_since_merger_flags(
    merger_type='Minor',
    windows_gyr=(0.75, 1.0, 1.25),
)

# Example: minor-merger windows using the default full catalog.
# fig, axes = plot_run_raw_merger_flags(
#     run=2,
#     expts=range(1, 9),
#     flags=minor_time_windows,
#     alpha_background=0.5,
#     show_legend=True,
# )

# Example: same default major-merger windows, but using a split catalog.
# fig, axes = plot_run_raw_merger_flags(
#     run=2,
#     expts=range(1, 9),
#     catalog_key='le_120x120',  # or 'gt_120x120'
#     flags=RAW_MERGER_FLAGS,
#     alpha_background=0.5,
#     show_legend=True,
# )



## Optional: Batch Runs

Uncomment this cell to generate the same time-since-merger overlays for several runs.



In [ ]:
# for run in [2, 3, 4, 5, 6, 7, 8]:
#     fig, axes = plot_run_raw_merger_flags(
#         run=run,
#         expts=range(1, 9),
#         flags=RAW_MERGER_FLAGS,
#         alpha_background=0.5,
#         show_legend=True,
#     )

